# 手撕 Decoder-Only Loss 实现

In [1]:
import torch
import math

## 数据和标签

In [2]:
batch_size = 1  # batch为多少条数据 
length = 4      # length 为 4

x = torch.randn(batch_size, 4, 512) #input :  batch_size, length, embd_dim
y = torch.randint(low=0, high=32000, size=(batch_size, 4), dtype=torch.long)

print(x.shape)
print(y.shape)
print(y)

torch.Size([1, 4, 512])

torch.Size([1, 4])

tensor([[17782, 11542, 29965, 20864]])

## Attention

In [3]:
q = torch.randn(512, 512)  
k = torch.randn(512, 512)
v = torch.randn(512, 512)
o = torch.randn(512, 512)

mask=torch.tril(torch.ones(1, 4, 4))
print(mask)

# scaled dot produc attention 
Q,K,V = x@q, x@k, x@v 
scores = Q@K.transpose(1,2) / math.sqrt(512.0)
scores = scores.masked_fill(mask == 0, float('-inf'))
weight = torch.nn.functional.softmax(scores, dim=2)
attn = weight@V
attn = attn@o
attn.shape

tensor([[[1., 0., 0., 0.],
         [1., 1., 0., 0.],
         [1., 1., 1., 0.],
         [1., 1., 1., 1.]]])

torch.Size([1, 4, 512])

## mlp

In [4]:
mlp_up = torch.randn(512, 1024)
mlp_down = torch.randn(1024, 512)
mlp = attn @ mlp_up @ mlp_down
mlp.shape

torch.Size([1, 4, 512])

## Output

In [5]:
lm_head = torch.randn(512, 32000) 
logits = mlp@lm_head
logits.shape

torch.Size([1, 4, 32000])

## Loss

In [6]:
# probs
probs = torch.softmax(logits, dim=2)
print(probs.shape) # model ouput prob
print(y)    # model lables

# Loss
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(probs.transpose(1, 2), y)
print(loss)

# pred
pred = torch.argmax(logits, dim=2)
print(pred) # model pred


torch.Size([1, 4, 32000])

tensor([[17782, 11542, 29965, 20864]])

tensor(10.3736)

tensor([[23469, 23469,  2696, 23469]])